# Test AMX HD (HV-AMX-CTRL-4EDH)

Diagnostic notebook for the **AMX HD** plugin driver. Connects to the real device
through the plugin's vendored `COM-HVAMX4EDH.dll` and runs the verification +
exploration sequence:

- identity probe (`GetProductID` / `GetHWType` / firmware) — confirm `HV-AMX-CTRL-4EDH`
- state encoding — confirm `STATE_ON = 0x0001` and `STATE_STANDBY = 0x0000` (the point that differed from the normal AMX)
- housekeeping (8 values), oscillator frequency, per-timer timing, config list (500 slots)

## Requirements

- **Windows** (the vendor DLL is Windows-only; on Linux `AMXHD()` raises `AMXHDPlatformError`).
- The AMX HD controller connected on a COM port.
- The loader auto-detects the repo root, so the notebook can be launched from anywhere.

> Inline mode (`process_backend=False`) is used so exceptions surface directly — easier to debug than the spawned worker.

## 1. Load the AMX HD driver from the plugin

Loads `AMXHD` directly from the plugin's vendored runtime (no ESIBD GUI needed).
This cell only imports — it does not touch hardware, so it is safe to run anywhere.

In [ ]:
import sys, types, importlib
from pathlib import Path

def _find_repo(start):
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / 'amx_hd' / 'vendor' / 'runtime').is_dir():
            return cand
    return start

REPO = _find_repo(Path.cwd())
RUNTIME = REPO / 'amx_hd' / 'vendor' / 'runtime'
assert RUNTIME.is_dir(), f'runtime not found (looked under {REPO})'

# Register the runtime dir as a package so the driver's relative imports resolve.
NS = '_nb_amx_hd_runtime'
pkg = types.ModuleType(NS)
pkg.__path__ = [str(RUNTIME)]
sys.modules[NS] = pkg

amx_hd_pkg = importlib.import_module(f'{NS}.amx_hd')
AMXHD = amx_hd_pkg.AMXHD

import platform
print('repo root:', REPO)
print('AMX HD driver class:', AMXHD)
print('platform:', platform.platform())
print('DLL present:', (RUNTIME / 'amx_hd' / 'vendor' / 'x64' / 'COM-HVAMX4EDH.dll').exists())

## 2. Connect

Edit `COM`, `BAUD`, `STREAM` for your setup. `process_backend=False` forces the
inline controller (direct calls, no worker process).

In [ ]:
COM = 5          # <- Windows COM port number (5 = COM5)
BAUD = 230400    # vendor default
STREAM = 0       # software-side stream (0..15)

device = AMXHD('hd_nb_test', com=COM, stream=STREAM, baudrate=BAUD, process_backend=False)
try:
    ok = device.connect(timeout_s=10)
    print('connect ->', ok)
except Exception as exc:
    print('connect FAILED:', type(exc).__name__, exc)
    print('(expected on non-Windows, or if the COM port / device is wrong)')

## 3. Identity probe

Confirm the connected unit is an HD controller. The expected product string
contains `HV-AMX-CTRL-4EDH` (the plugin's `_EXPECTED_PRODUCT_TOKENS`).

In [ ]:
try:
    info = device.get_product_info(timeout_s=10)
    import json
    print(json.dumps(info, indent=2, default=str))
    pid = str(info.get('product_id', ''))
    print('\nLooks like HD controller?', '4EDH' in pid.upper() or 'AMX' in pid.upper())
except Exception as exc:
    print('identity probe FAILED:', type(exc).__name__, exc)

## 4. State encoding (the key HD check)

The HD state encoding differs from the normal AMX:
- `0x0001` = `STATE_ON`
- `0x0000` = `STATE_STANDBY` (on the normal AMX `0x0000` was `STATE_ON`)

This cell reads `collect_housekeeping()` and prints the main state. Compare
standby (device OFF/not enabled) vs ON (device enabled) to confirm the codes.

In [ ]:
try:
    snap = device.collect_housekeeping(timeout_s=15)
    ms = snap['main_state']
    print('main_state:', ms['name'], '(' + str(ms['hex']) + ')')
    print('device_enabled:', snap['device_enabled'])
    print('device_state flags:', snap['device_state']['flags'])
    print('controller_state flags:', snap['controller_state']['flags'])
    print()
    print('VERDICT:')
    if ms['name'] == 'STATE_ON':
        print('  ON reported as STATE_ON -> matches HD header (0x0001). OK.')
    elif ms['name'] == 'STATE_STANDBY':
        print('  Standby reported as STATE_STANDBY (0x0000) -> NOT treated as ON. OK.')
    else:
        print('  Unexpected main_state:', ms['name'], '- investigate.')
except Exception as exc:
    print('state probe FAILED:', type(exc).__name__, exc)

## 5. Housekeeping (8 values), frequency, timers, configs

The HD housekeeping returns 8 voltages (vs 4 on normal). Frequency uses
oscillator-0. Timers replace pulsers (count queried at runtime). Configs go up
to 500 slots.

In [ ]:
try:
    if 'snap' not in globals():
        snap = device.collect_housekeeping(timeout_s=15)
    hk = snap.get('housekeeping', {})
    print('Housekeeping (8 values):')
    for k, v in hk.items():
        print(f'  {k:16s} = {v}')
except Exception as exc:
    print('housekeeping FAILED:', type(exc).__name__, exc)

print()
try:
    f_khz = device.get_frequency_khz(timeout_s=10)
    print(f'oscillator-0 frequency: {f_khz:.4g} kHz')
except Exception as exc:
    print('frequency read FAILED:', type(exc).__name__, exc)

print()
try:
    timers = snap.get('pulsers', [])
    print(f'timers reported: {len(timers)}')
    for t in timers:
        print(f"  timer {t['pulser']}: width_ticks={t['width_ticks']} delay_ticks={t['delay_ticks']} burst={t['burst']}")
except Exception as exc:
    print('timer read FAILED:', type(exc).__name__, exc)

print()
try:
    configs = device.list_configs(include_empty=False, timeout_s=15)
    print(f'non-empty config slots: {len(configs)} / 500')
    for c in configs[:15]:
        print(' ', c)
    if len(configs) > 15:
        print(f'  ... (+{len(configs) - 15} more)')
except Exception as exc:
    print('config list FAILED:', type(exc).__name__, exc)

## 6. (Optional) Set frequency / timer width round-trip

Uncomment to actually write values. Keep amplitudes/setpoints within the device
limits; the HD controller does not set HV amplitude (external PSUs do).

In [ ]:
# Round-trip a frequency (kHz) and a timer-0 width (ticks).
# try:
#     device.set_frequency_khz(2.0, timeout_s=10)
#     print('set 2.0 kHz -> readback', device.get_frequency_khz(timeout_s=10), 'kHz')
#     device.set_pulser_width_ticks(0, 500, timeout_s=10)
#     print('timer0 width readback:', device.get_pulser_width_ticks(0, timeout_s=10))
# except Exception as exc:
#     print('round-trip FAILED:', type(exc).__name__, exc)
print('(uncomment the block above to write values to the device)')

## 7. Disconnect

Always run this last to release the COM port.

In [ ]:
try:
    disconnected = device.shutdown(disable_device=True, timeout_s=10)
    print('shutdown ->', disconnected)
except Exception as exc:
    print('shutdown FAILED:', type(exc).__name__, exc)
    try:
        device.disconnect(timeout_s=5)
    except Exception:
        pass